# 03 Baseline Models

This notebook establishes a performance floor for selling price prediction using four baseline models on the full encoded feature set (~920 features). The goal is to understand how far standard approaches can take us before introducing gradient boosting.

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## Data Loading and Train/Test Split

We use the fully cleaned and one-hot encoded dataset produced by `01_data_cleaning.ipynb`. The split is 80/20 with a fixed random state for reproducibility. This same split will be reused in the gradient boosting notebook for an apples-to-apples comparison.

In [10]:
df = pd.read_csv('data/cleaned/car_prices_cleaned.csv')
print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

X = df.drop('sellingprice', axis=1)
y = df['sellingprice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"Features:     {X_train.shape[1]}")

Dataset: 545,433 rows x 920 columns
Training set: 436,346 samples
Test set:     109,087 samples
Features:     919


## Model Training and Evaluation

Each model is trained on the training set and evaluated on the held-out test set using two metrics:
- **MAE** (Mean Absolute Error): average dollar amount the prediction is off by.
- **R²**: proportion of variance in selling price explained by the model (1.0 = perfect).

In [11]:
results = []

def evaluate_model(model, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'MAE ($)': round(mae, 2), 'R²': round(r2, 4)})
    print(f"{name:25s} | MAE: ${mae:,.2f}  |  R²: {r2:.4f}")
    return model

### Linear Regression

In [12]:
lr = evaluate_model(LinearRegression(), 'Linear Regression')

Linear Regression         | MAE: $969.87  |  R²: 0.9704


### Ridge Regression

L2 regularization to penalize large coefficients and reduce potential overfitting with 920 features.

In [13]:
ridge = evaluate_model(Ridge(alpha=1.0), 'Ridge Regression')

Ridge Regression          | MAE: $969.70  |  R²: 0.9705


### Lasso Regression

L1 regularization, which can drive some feature coefficients to zero -- effectively performing feature selection.

In [14]:
lasso = evaluate_model(Lasso(alpha=0.1), 'Lasso Regression')

Lasso Regression          | MAE: $968.98  |  R²: 0.9704


### Random Forest Regressor

An ensemble of decision trees that can capture non-linear relationships between features.

In [15]:
rf = evaluate_model(
    RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Random Forest'
)

Random Forest             | MAE: $954.36  |  R²: 0.9712


## Comparison Table

In [16]:
results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values('MAE ($)')
print(f"\nAverage selling price: ${y.mean():,.2f}")
print(f"Baseline comparison (test set):\n")
results_df


Average selling price: $13,725.53
Baseline comparison (test set):



,MAE ($),R²
Model,,
Random Forest,954.36,0.9712
Lasso Regression,968.98,0.9704
Ridge Regression,969.70,0.9705
Linear Regression,969.87,0.9704


In [17]:
import json

baseline_output = {row['Model']: {'MAE': row['MAE ($)'], 'R2': row['R²']} for row in results}
with open('data/baseline_results.json', 'w') as f:
    json.dump(baseline_output, f, indent=2)

print("Baseline results saved to data/baseline_results.json")

Baseline results saved to data/baseline_results.json


## Conclusion

All four baselines achieve strong R² scores, confirming that the feature set captures most of the variance in selling price. The linear models and Random Forest perform comparably on this dataset, which is expected given that MMR (an independent market valuation) is included as a feature and likely dominates the prediction.

The baseline MAE establishes the floor we need to beat. In the next notebook, we apply Gradient Boosting to see how much further we can reduce prediction error by better capturing non-linear interactions across the full feature set.